# PET hydrolysis kinetics: model fitting, uncertainty and identifiability

**JACS Au | Computational supplement — Notebook 04**

This notebook reconstructs the student's kinetic-model reaction network and documents the additional reviewer-directed analyses. The original MATLAB rate-constant estimates are preserved in the input workbook; Python fitting uses **effective kinetic parameters** for the same product-forming network, rather than treating non-identifiable individual association/formation constants as independent mechanistic quantities.

**Workflow:** data → 55-condition fits → representative profiles → uncertainty → branching ratios → multi-start identifiability → alternative pathways → exports.

**Scope:** Descriptive apparent kinetics, not unique mechanistic rate identification. The supplementary analyses include an enzyme-deactivation parameter `k8` that is not explicitly documented in the manuscript's equations 5–11. Check this distinction against the source MATLAB model before citing full equivalence.

## 1. Kinetic network and assumptions

The empirical model follows BHET, MHET and TPA formation, with subsequent BHET → MHET → TPA hydrolysis. Substrate is approximated as constant; the fitted effective formation coefficients combine the inseparable original parameters `k1`–`k5`. The model includes optional first-order enzyme deactivation `k8`. Time is in hours; measured product concentration is in µM.

## 2. Experimental data and fitting across 55 conditions

The workbook provides experimental replicate profiles, conditions and the original MATLAB results. Each replicate is fitted independently, followed by a condition-level summary. The reported Python rates are **effective** rates and should not be substituted numerically for the original MATLAB `k1`–`k5` values.

In [ ]:
# CELL 1 — Import the entire experimental dataset from Excel
# -----------------------------------------------------------
# Only this Excel workbook is required. Each row is one measured time point
# from one experimental replicate. Concentrations are already calibrated to µM.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

EXCEL_FILE = Path("data/PET_kinetic_data.xlsx")

data = pd.read_excel(EXCEL_FILE, sheet_name="Experiment_Data")
conditions_table = pd.read_excel(EXCEL_FILE, sheet_name="Conditions")
matlab_results = pd.read_excel(EXCEL_FILE, sheet_name="MATLAB_Results")

# Keep the product order used by the kinetic model: BHET, MHET, TPA.
PRODUCT_COLS = ["BHET_uM", "MHET_uM", "TPA_uM"]

print(f"Rows imported:       {len(data):,}")
print(f"Experimental sets:   {data['Condition_ID'].nunique()}")
print(f"Source files:        {data['Source_File'].nunique()}")
print(f"Replicates (total):  {data[['Condition_ID','Replicate']].drop_duplicates().shape[0]}")
display(data.head())

# Helper: extract one condition as replicate-wise time courses.
def get_condition(condition_id):
    d = data.loc[data["Condition_ID"].eq(condition_id)].copy()
    if d.empty:
        raise ValueError(f"Condition_ID {condition_id} was not found.")

    first = d.iloc[0]
    cond = {
        "Condition_ID": int(condition_id),
        "Source_File": first["Source_File"],
        "Enzyme_uM": float(first["Enzyme_uM"]),
        "Substrate_discs_L": float(first["Substrate_discs_L"]),
        "EG_fraction": float(first["EG_fraction"]),
        "EG_volpct": float(first["EG_volpct"]),
        "Temperature_C": float(first["Temperature_C"]),
        "Temperature_K": float(first["Temperature_K"]),
        "pH": float(first["pH"]),
    }

    reps = []
    for rep_id, g in d.groupby("Replicate", sort=True):
        g = g.sort_values("Time_h")
        reps.append({
            "Replicate": int(rep_id),
            "time": g["Time_h"].to_numpy(float),
            "observed": g[PRODUCT_COLS].to_numpy(float),
        })
    return cond, reps


In [ ]:
# CELL 2 — Student kinetic model + nonlinear fitting
# --------------------------------------------------
# The student's ODEs can be written in an identifiable equivalent form:
#
# dB/dt   = beta_B * E * S - k6 * B
# dM/dt   = beta_M * E * S + k6 * B - k7 * M
# dTPA/dt = beta_T * E * S + k7 * M
# dE/dt   = -k8 * E
#
# where beta_B, beta_M and beta_T contain the original non-identifiable
# combination k_i * [(k4/k5)/(1-k4/k5)]. Thus, this retains the exact
# concentration-time behaviour of ODE_ModelSimple.m while avoiding arbitrary
# individual values of k1-k5.

PARAM_NAMES = ["beta_B", "beta_M", "beta_T", "k6", "k7", "k8"]

# Starting values derived from the student's original MATLAB starting guess.
K4_0 = 0.1 / 2.0
q0 = K4_0 / (1.0 - K4_0)
THETA0 = np.array([
    100.0*q0, 35.0*q0, 0.5*q0,  # beta_B, beta_M, beta_T
    0.01, 0.01, 0.0             # k6, k7, k8
], dtype=float)

def rhs_student(t, y, theta, substrate_discs_L):
    """Exact reduced form of the student's active ODE model."""
    B, M, TPA, E = y
    beta_B, beta_M, beta_T, k6, k7, k8 = theta
    S = substrate_discs_L

    return [
        beta_B * E * S - k6 * B,
        beta_M * E * S + k6 * B - k7 * M,
        beta_T * E * S + k7 * M,
        -k8 * E,
    ]

def simulate(theta, time, cond):
    """Integrate the ODE model at requested experimental times."""
    time = np.asarray(time, float)
    y0 = [0.0, 0.0, 0.0, cond["Enzyme_uM"]]

    sol = solve_ivp(
        lambda t, y: rhs_student(t, y, theta, cond["Substrate_discs_L"]),
        (0.0, float(time.max())),
        y0,
        t_eval=time,
        method="RK45",
        rtol=1e-7,
        atol=1e-9,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol.y.T

def residual(theta, time, observed, cond):
    """Residuals for simultaneous BHET, MHET and TPA fitting."""
    predicted = simulate(theta, time, cond)[:, :3]
    return (predicted - observed).ravel()

def fit_one_condition(condition_id):
    """
    Reproduce the MATLAB workflow:
      1) fit each replicate independently;
      2) average fitted parameters across replicates;
      3) report replicate-to-replicate SD as parameter uncertainty.
    """
    cond, reps = get_condition(condition_id)
    fitted = []
    metrics = []

    for rep in reps:
        out = least_squares(
            residual,
            THETA0,
            args=(rep["time"], rep["observed"], cond),
            # No positivity bounds are imposed here because the original MATLAB
            # fitting was unconstrained. Bounds can be introduced in Stage 2.
            max_nfev=5000,
        )

        pred = simulate(out.x, rep["time"], cond)[:, :3]
        obs = rep["observed"]
        rmse = np.sqrt(np.mean((pred - obs)**2))
        sse = np.sum((pred - obs)**2)
        sst = np.sum((obs - obs.mean(axis=0, keepdims=True))**2)
        r2 = 1.0 - sse/sst if sst > 0 else np.nan

        fitted.append(out.x)
        metrics.append((rep["Replicate"], rmse, r2, out.success))

    fitted = np.vstack(fitted)
    ddof = 1 if len(fitted) > 1 else 0

    result = {
        **cond,
        "n_replicates": len(fitted),
        **{p: fitted[:, i].mean() for i, p in enumerate(PARAM_NAMES)},
        **{f"sd_{p}": fitted[:, i].std(ddof=ddof) for i, p in enumerate(PARAM_NAMES)},
        "RMSE_uM": np.mean([m[1] for m in metrics]),
        "R2": np.mean([m[2] for m in metrics]),
    }
    return result, fitted

# Fit all 55 conditions.
fit_rows = []
replicate_parameter_store = {}

for condition_id in sorted(data["Condition_ID"].unique()):
    result, rep_pars = fit_one_condition(int(condition_id))
    fit_rows.append(result)
    replicate_parameter_store[int(condition_id)] = rep_pars

fit_results = pd.DataFrame(fit_rows)


In [ ]:
# CELL 3 — Clear fitted-results table
# -----------------------------------
# Mean fitted parameters and replicate-to-replicate SD are reported together.
# This gives a direct first view of parameter uncertainty for every condition.

result_columns = [
    "Condition_ID", "Source_File", "n_replicates",
    "Enzyme_uM", "Substrate_discs_L", "EG_volpct",
    "Temperature_C", "pH",
    "beta_B", "sd_beta_B",
    "beta_M", "sd_beta_M",
    "beta_T", "sd_beta_T",
    "k6", "sd_k6",
    "k7", "sd_k7",
    "k8", "sd_k8",
    "RMSE_uM", "R2",
]

fitted_results_table = fit_results[result_columns].copy()
display(
    fitted_results_table.style.format({
        "beta_B": "{:.4g}", "sd_beta_B": "{:.3g}",
        "beta_M": "{:.4g}", "sd_beta_M": "{:.3g}",
        "beta_T": "{:.4g}", "sd_beta_T": "{:.3g}",
        "k6": "{:.4g}", "sd_k6": "{:.3g}",
        "k7": "{:.4g}", "sd_k7": "{:.3g}",
        "k8": "{:.4g}", "sd_k8": "{:.3g}",
        "RMSE_uM": "{:.2f}", "R2": "{:.3f}",
    })
)

# Optional export of the Python-fitted table:
#fitted_results_table.to_excel("PET_Stage1_fitted_results.xlsx", index=False)


## 3. Representative kinetic profiles: 0% and 40% EG

Compare fitted BHET, MHET and TPA trajectories with experimental replicate means and standard deviations at 250 nM enzyme, four PET discs/mL, 60 °C and pH 10.

In [ ]:
# CELL 4 — Product-profile plot with experimental uncertainty
# ------------------------------------------------------------
# Error bars = mean ± 1 standard deviation across experimental replicates
# at each time point. Solid lines are the kinetic model evaluated using the
# mean fitted parameters for that condition.
#
# The default comparison exactly matches the manuscript-style example:
# 250 nM enzyme, 4 discs/mL, pH 10, 60 °C, comparing 40% vs 0% EG.

def find_condition(enzyme_uM=0.25, substrate_discs_mL=4, eg_volpct=40,
                   temperature_C=60, pH=10):
    q = conditions_table[
        np.isclose(conditions_table["Enzyme_uM"], enzyme_uM) &
        np.isclose(conditions_table["Substrate_discs_mL"], substrate_discs_mL) &
        np.isclose(conditions_table["EG_volpct"], eg_volpct) &
        np.isclose(conditions_table["Temperature_C"], temperature_C) &
        np.isclose(conditions_table["pH"], pH)
    ]
    if q.empty:
        raise ValueError("No condition matches the requested settings.")
    return int(q.iloc[0]["Condition_ID"])

def mean_sd_profile(condition_id):
    d = data.loc[data["Condition_ID"].eq(condition_id)].copy()
    # Replicate statistics at every time point
    stats = (
        d.groupby("Time_h")[PRODUCT_COLS]
         .agg(["mean", "std"])
         .sort_index()
    )
    return stats

def theta_from_results(condition_id):
    r = fit_results.loc[fit_results["Condition_ID"].eq(condition_id)].iloc[0]
    return np.array([r[p] for p in PARAM_NAMES], dtype=float)

def plot_product_profiles(condition_ids, panel_titles=None):
    n = len(condition_ids)
    fig, axes = plt.subplots(1, n, figsize=(6.3*n, 4.8), squeeze=False)
    axes = axes.ravel()

    # Fixed product colors chosen to resemble the manuscript figure.
    colors = {"BHET": "#E6550D", "MHET": "#F0A202", "TPA": "#7B3294"}

    for ax, condition_id, panel_title in zip(
        axes, condition_ids,
        panel_titles if panel_titles is not None else [None]*n
    ):
        cond, _ = get_condition(condition_id)
        stats = mean_sd_profile(condition_id)
        theta = theta_from_results(condition_id)

        t_obs = stats.index.to_numpy(float)
        t_grid = np.linspace(0, t_obs.max(), 300)
        pred_uM = simulate(theta, t_grid, cond)[:, :3]

        for j, product in enumerate(["BHET", "MHET", "TPA"]):
            mean_uM = stats[(f"{product}_uM", "mean")].to_numpy(float)
            sd_uM = stats[(f"{product}_uM", "std")].fillna(0).to_numpy(float)

            # Convert µM -> mM to match the manuscript presentation.
            ax.errorbar(
                t_obs, mean_uM/1000.0, yerr=sd_uM/1000.0,
                fmt="o", ms=4, capsize=3, elinewidth=1,
                color=colors[product], alpha=0.85
            )
            ax.plot(
                t_grid, pred_uM[:, j]/1000.0,
                lw=1.8, color=colors[product], label=product
            )

        title = panel_title or f"EG={cond['EG_volpct']:.0f}%"
        ax.set_title(title, loc="left", fontsize=13)
        ax.set_xlabel("t / h")
        ax.set_ylabel("Concentration / mM")
        ax.spines["top"].set_visible(True)
        ax.spines["right"].set_visible(True)

        # Show a compact set of fitted parameters.
        beta_B, beta_M, beta_T, k6, k7, k8 = theta
        txt = (
            rf"$\beta_B$={beta_B:.3g}" + "\n" +
            rf"$\beta_T$={beta_T:.3g}" + "\n" +
            rf"$k_6$={k6:.3g} h$^{{-1}}$"
        )
        ax.text(
            0.62, 0.72, txt, transform=ax.transAxes,
            fontsize=10, va="top"
        )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    plt.show()

# Conditions used in the manuscript-style comparison
cid_40 = find_condition(eg_volpct=40)
cid_0  = find_condition(eg_volpct=0)

plot_product_profiles(
    [cid_40, cid_0],
    panel_titles=["EG=40%", "EG=0%"]
)


## 4. Parameter uncertainty and bootstrap prediction bands

The 55-condition intervals use replicate-level **t intervals from the original MATLAB parameter summaries**. Separately, 500 nonparametric bootstrap refits are performed for each representative EG condition, resampling complete experimental replicate profiles. This section uses the analytically solved equivalent effective-rate equations for faster repeated fits.

In [ ]:
# SETUP — import data and define the unchanged Stage-1 kinetic model
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.special import exprel
from scipy.stats import t as student_t, chi2

EXCEL_FILE = Path("data/PET_kinetic_data.xlsx")
data = pd.read_excel(EXCEL_FILE, sheet_name="Experiment_Data")
conditions = pd.read_excel(EXCEL_FILE, sheet_name="Conditions")
matlab_results = pd.read_excel(EXCEL_FILE, sheet_name="MATLAB_Results")

PRODUCT_COLS = ["BHET_uM", "MHET_uM", "TPA_uM"]
PARAMS = ["beta_B", "beta_M", "beta_T", "k6", "k7", "k8"]

def get_condition(cid):
    d = data.loc[data["Condition_ID"].eq(cid)].copy()
    f = d.iloc[0]
    cond = {
        "Condition_ID": int(cid), "Source_File": f["Source_File"],
        "Enzyme_uM": float(f["Enzyme_uM"]), "Substrate_discs_L": float(f["Substrate_discs_L"]),
        "EG_volpct": float(f["EG_volpct"]), "Temperature_C": float(f["Temperature_C"]), "pH": float(f["pH"])
    }
    reps = []
    for rid, g in d.groupby("Replicate", sort=True):
        g = g.sort_values("Time_h")
        reps.append({"Replicate": int(rid), "time": g["Time_h"].to_numpy(float), "observed": g[PRODUCT_COLS].to_numpy(float)})
    return cond, reps

def F(a, b, t):
    t = np.asarray(t, float)
    return np.exp(-b*t) * t * exprel(-(a-b)*t)

def G(k, t):
    t = np.asarray(t, float)
    return t * exprel(-k*t)

def simulate(theta, time, cond):
    beta_B, beta_M, beta_T, k6, k7, k8 = np.asarray(theta, float)
    t = np.asarray(time, float)
    E0, S = cond["Enzyme_uM"], cond["Substrate_discs_L"]
    AB, AM, AT = beta_B*S*E0, beta_M*S*E0, beta_T*S*E0
    B = AB * F(k6, k8, t)
    d = k6-k8
    if abs(d) < 1e-6*(1+abs(k6)+abs(k8)):
        k = 0.5*(k6+k8); h = 1e-5*(1+abs(k))
        J = -(F(k7,k+h,t)-F(k7,k-h,t))/(2*h)
    else:
        J = (F(k7,k8,t)-F(k7,k6,t))/d
    M = AM*F(k7,k8,t) + k6*AB*J
    total = (AB+AM+AT)*G(k8,t)
    TPA = total-B-M
    E = E0*np.exp(-k8*t)
    return np.column_stack([B,M,TPA,E])

def residual(theta, reps, cond):
    rr=[]
    try:
        for r in reps:
            pred=simulate(theta,r["time"],cond)[:,:3]
            if not np.all(np.isfinite(pred)) or np.max(np.abs(pred))>1e12:
                raise ValueError
            rr.append((pred-r["observed"]).ravel())
        return np.concatenate(rr)
    except Exception:
        return np.ones(sum(r["observed"].size for r in reps))*1e10

def fit_model(cond, reps, x0, max_nfev=700):
    out=least_squares(residual,np.asarray(x0,float),args=(reps,cond),max_nfev=max_nfev,xtol=1e-9,ftol=1e-9,gtol=1e-9)
    res=residual(out.x,reps,cond); sse=float(np.sum(res**2)); n=len(res)
    obs=np.concatenate([r["observed"].ravel() for r in reps]); sst=float(np.sum((obs-obs.mean())**2))
    return out,{"SSE":sse,"RMSE_uM":np.sqrt(sse/n),"R2":1-sse/sst,"n":n}

def matlab_to_effective(source_file):
    r=matlab_results.loc[matlab_results["File"].eq(source_file)].iloc[0]
    K=float(r["k_4"]/r["k_5"]); q=K/(1-K)
    return np.array([r["k_1"]*q,r["k_2"]*q,r["k_3"]*q,r["k_6"],r["k_7"],r["k_8"]],float)

# Manuscript comparison conditions: 250 nM enzyme, 4 discs/mL, 60 °C, pH 10, EG=40% vs 0%.
key = conditions[
    np.isclose(conditions["Enzyme_uM"],0.25) & np.isclose(conditions["Substrate_discs_mL"],4) &
    np.isclose(conditions["Temperature_C"],60) & np.isclose(conditions["pH"],10) & conditions["EG_volpct"].isin([40,0])
].sort_values("EG_volpct",ascending=False)
key_ids=key["Condition_ID"].astype(int).tolist()
print("Key condition IDs:", key_ids)


In [ ]:
# CELL 5 — Parameter uncertainty + 500 bootstrap fits
# ---------------------------------------------------
# First, estimate 95% CIs for the original k1-k8 across all 55 conditions from
# replicate-to-replicate SD. Then perform 500 cluster-bootstrap fits for each of
# the two manuscript conditions, resampling complete replicate time profiles.

all_ci=[]
for _,c in conditions.iterrows():
    r=matlab_results.loc[matlab_results["File"].eq(c["Source_File"])].iloc[0]
    n=int(c["n_replicates"]); tc=student_t.ppf(.975,n-1)
    row={"Condition_ID":int(c["Condition_ID"]),"Source_File":c["Source_File"],"n_replicates":n}
    for j in range(1,9):
        mean=float(r[f"k_{j}"]); sd=float(r[f"sdk_{j}"]); se=sd/np.sqrt(n)
        row[f"k{j}_mean"]=mean; row[f"k{j}_CI_low"]=mean-tc*se; row[f"k{j}_CI_high"]=mean+tc*se
    all_ci.append(row)
all_condition_CI=pd.DataFrame(all_ci)
zero_table=pd.DataFrame({"parameter":[f"k{j}" for j in range(1,9)],"conditions_CI_includes_zero":[int(((all_condition_CI[f"k{j}_CI_low"]<=0)&(all_condition_CI[f"k{j}_CI_high"]>=0)).sum()) for j in range(1,9)]})
display(zero_table)

base={}
for cid in key_ids:
    cond,reps=get_condition(cid); x0=matlab_to_effective(cond["Source_File"]); out,met=fit_model(cond,reps,x0,1200); base[cid]=(cond,reps,out.x,met)

N_BOOT=500; rng=np.random.default_rng(20260918); rows=[]
for cid in key_ids:
    cond,reps,best,_=base[cid]; nrep=len(reps)
    for b in range(N_BOOT):
        sampled=[reps[i] for i in rng.integers(0,nrep,size=nrep)]
        out,met=fit_model(cond,sampled,best,400)
        row={"Condition_ID":cid,"EG_volpct":cond["EG_volpct"],"bootstrap":b+1}
        row.update({p:out.x[i] for i,p in enumerate(PARAMS)})
        row["k1_over_k3"]=out.x[0]/out.x[2] if abs(out.x[2])>1e-12 else np.nan
        rows.append(row)
bootstrap=pd.DataFrame(rows)

summary=[]
for (cid,eg),g in bootstrap.groupby(["Condition_ID","EG_volpct"]):
    row={"Condition_ID":cid,"EG_volpct":eg,"n_bootstrap":len(g)}
    for p in PARAMS+["k1_over_k3"]:
        v=g[p].replace([np.inf,-np.inf],np.nan).dropna().to_numpy(float)
        row[p+"_median"]=np.median(v); row[p+"_CI_low"]=np.percentile(v,2.5); row[p+"_CI_high"]=np.percentile(v,97.5)
    summary.append(row)
bootstrap_summary=pd.DataFrame(summary)
display(bootstrap_summary.round(4))

fig,axes=plt.subplots(2,3,figsize=(11,6.4))
for ax,p in zip(axes.ravel(),PARAMS):
    for cid in key_ids:
        g=bootstrap.loc[bootstrap["Condition_ID"].eq(cid),p]; eg=bootstrap.loc[bootstrap["Condition_ID"].eq(cid),"EG_volpct"].iloc[0]
        ax.hist(g,bins=28,alpha=.45,label=f"EG={eg:.0f}%")
    ax.set_title(p); ax.set_ylabel("Count")
axes[0,0].legend(frameon=False); fig.suptitle("500-bootstrap parameter distributions"); fig.tight_layout(rect=[0,0,1,.95]); plt.show()

# Product profiles with experimental SD and bootstrap prediction bands
fig,axes=plt.subplots(1,2,figsize=(12,4.6))
for ax,cid in zip(axes,key_ids):
    cond,reps,_,_=base[cid]; d=data[data["Condition_ID"].eq(cid)]; stats=d.groupby("Time_h")[PRODUCT_COLS].agg(["mean","std"]).sort_index(); t=stats.index.to_numpy(float); grid=np.linspace(0,t.max(),200)
    g=bootstrap[bootstrap["Condition_ID"].eq(cid)].dropna(subset=PARAMS); samp=g.iloc[np.linspace(0,len(g)-1,200,dtype=int)]
    preds=np.stack([simulate(r[PARAMS].to_numpy(float),grid,cond)[:,:3] for _,r in samp.iterrows()]); lo,med,hi=np.percentile(preds,[2.5,50,97.5],axis=0)
    for j,name in enumerate(["BHET","MHET","TPA"]):
        mean=stats[(name+"_uM","mean")].to_numpy()/1000; sd=stats[(name+"_uM","std")].fillna(0).to_numpy()/1000
        ax.errorbar(t,mean,yerr=sd,fmt='o',ms=4,capsize=3); ax.plot(grid,med[:,j]/1000,label=name); ax.fill_between(grid,lo[:,j]/1000,hi[:,j]/1000,alpha=.15)
    ax.set_title(f"EG={cond['EG_volpct']:.0f}%"); ax.set_xlabel("t / h"); ax.set_ylabel("Concentration / mM")
axes[0].legend(frameon=False); fig.suptitle("Fits with 95% bootstrap prediction bands"); fig.tight_layout(rect=[0,0,1,.95]); plt.show()


## 5. Branching-ratio analysis

Because the common enzyme–substrate factor cancels, `beta_B / beta_T` corresponds to `k1 / k3` within the specified model. Ratios from one local best-fit basin are **not** evidence of global identifiability, particularly at 0% EG.

In [ ]:
# CELL 6 — Confidence interval for the mechanistic k1/k3 ratio
# ----------------------------------------------------------------
# beta_B/beta_T = k1/k3 exactly, because the common k4/k5 factor cancels.
ratio_table=bootstrap_summary[["Condition_ID","EG_volpct","k1_over_k3_median","k1_over_k3_CI_low","k1_over_k3_CI_high"]].copy()
display(ratio_table.round(3))
fig,ax=plt.subplots(figsize=(6,4.2)); arr=[]; labels=[]
for cid in key_ids:
    v=bootstrap.loc[bootstrap["Condition_ID"].eq(cid),"k1_over_k3"].replace([np.inf,-np.inf],np.nan).dropna(); lo,hi=np.percentile(v,[.5,99.5]); arr.append(v.clip(lo,hi)); labels.append(f"EG={bootstrap.loc[bootstrap['Condition_ID'].eq(cid),'EG_volpct'].iloc[0]:.0f}%")
ax.boxplot(arr,labels=labels,showfliers=False); ax.set_ylabel(r"$k_1/k_3$ (= $\beta_B/\beta_T$)"); ax.set_title("Bootstrap uncertainty in mechanistic ratio"); plt.show()


## 6. Practical parameter identifiability

Use 80 parameter starts per representative condition and inspect solutions within 1% of the lowest SSE. Profile likelihood is an additional diagnostic; search-grid boundaries and physically implausible negative deactivation rates require cautious interpretation.

In [ ]:
# CELL 7 — Multi-start and profile-likelihood identifiability
rng=np.random.default_rng(20260917); rows=[]; N_START=80
for cid in key_ids:
    cond,reps,best,_=base[cid]; scale=np.maximum(np.abs(best),np.array([.1,.1,.05,.05,.02,.05])); starts=[best.copy()]+[best+rng.normal(size=6)*1.5*scale for _ in range(N_START-1)]
    for i,x0 in enumerate(starts):
        out,met=fit_model(cond,reps,x0,900); row={"Condition_ID":cid,"EG_volpct":cond["EG_volpct"],"start":i+1,"SSE":met["SSE"]}; row.update({p:out.x[j] for j,p in enumerate(PARAMS)}); rows.append(row)
multistart=pd.DataFrame(rows)
ms=[]
for (cid,eg),g in multistart.groupby(["Condition_ID","EG_volpct"]):
    mn=g["SSE"].min(); near=g[g["SSE"]<=1.01*mn]; row={"Condition_ID":cid,"EG_volpct":eg,"n_starts":len(g),"n_within_1pct_SSE":len(near)}
    for p in PARAMS: row[p+"_min"]=near[p].min(); row[p+"_max"]=near[p].max()
    ms.append(row)
multistart_summary=pd.DataFrame(ms); display(multistart_summary.round(4))

fig,axes=plt.subplots(2,3,figsize=(11,6.4))
for ax,p in zip(axes.ravel(),PARAMS):
    for cid in key_ids:
        g=multistart[multistart["Condition_ID"].eq(cid)].copy(); g["relSSE"]=g["SSE"]/g["SSE"].min(); eg=g["EG_volpct"].iloc[0]; ax.scatter(g[p],g["relSSE"],s=18,alpha=.6,label=f"EG={eg:.0f}%")
    ax.axhline(1.01,ls='--',lw=1); ax.set_xlabel(p); ax.set_ylabel("SSE / best SSE"); ax.set_ylim(.995,1.15)
axes[0,0].legend(frameon=False); fig.suptitle("Multi-start practical identifiability"); fig.tight_layout(rect=[0,0,1,.95]); plt.show()

# Profile likelihood scans
prof=[]
for cid in key_ids:
    cond,reps,best,met=base[cid]; sse0=met["SSE"]; n=met["n"]; threshold=sse0+chi2.ppf(.95,1)*(sse0/max(n-6,1)); bg=bootstrap[bootstrap["Condition_ID"].eq(cid)]
    for j,p in enumerate(PARAMS):
        q1,q99=np.percentile(bg[p],[1,99]); span=max(q99-q1,1e-6); grid=np.linspace(q1-.5*span,q99+.5*span,31); free=[k for k in range(6) if k!=j]; xf=best[free].copy()
        for fixed in grid:
            def rf(x): th=best.copy(); th[j]=fixed; th[free]=x; return residual(th,reps,cond)
            out=least_squares(rf,xf,max_nfev=600,xtol=1e-9,ftol=1e-9,gtol=1e-9); xf=out.x; th=best.copy(); th[j]=fixed; th[free]=xf; sse=float(np.sum(residual(th,reps,cond)**2)); prof.append({"Condition_ID":cid,"EG_volpct":cond["EG_volpct"],"parameter":p,"fixed_value":fixed,"SSE":sse,"SSE_min":sse0,"threshold95":threshold})
profile=pd.DataFrame(prof)
ps=[]
for (cid,eg,p),g in profile.groupby(["Condition_ID","EG_volpct","parameter"]):
    a=g[g["SSE"]<=g["threshold95"]]; ps.append({"Condition_ID":cid,"EG_volpct":eg,"parameter":p,"profile_CI_low":a["fixed_value"].min() if len(a) else np.nan,"profile_CI_high":a["fixed_value"].max() if len(a) else np.nan,"hits_grid_edge":bool(len(a) and (np.isclose(a["fixed_value"].min(),g["fixed_value"].min()) or np.isclose(a["fixed_value"].max(),g["fixed_value"].max())))})
profile_summary=pd.DataFrame(ps); display(profile_summary.round(4))


## 7. Alternative reaction pathways

Four computational formulations are tested, but **only three appear in manuscript Table S8**: notebook `M1_full` → manuscript M1; notebook `M3_no_direct_TPA` → manuscript M2; notebook `M4_sequential_only` → manuscript M3. The notebook-only `M2_no_deactivation` is an exploratory diagnostic. Reported BIC comparisons are **ΔBIC relative to the best model for each condition**, not raw BIC.

In [ ]:
# CELL 8 — Alternative kinetic schemes across all 55 conditions
models={"M1_full":[0,1,2,3,4,5],"M2_no_deactivation":[0,1,2,3,4],"M3_no_direct_TPA":[0,1,3,4,5],"M4_sequential_only":[0,3,4,5]}
rows=[]
for _,c in conditions.iterrows():
    cid=int(c["Condition_ID"]); cond,reps=get_condition(cid); x0=matlab_to_effective(cond["Source_File"]); full,_=fit_model(cond,reps,x0,1000); best=full.x; obs=np.concatenate([r["observed"].ravel() for r in reps]); n=len(obs)
    for name,free in models.items():
        fixed=best.copy()
        if name=="M2_no_deactivation": fixed[5]=0
        elif name=="M3_no_direct_TPA": fixed[2]=0
        elif name=="M4_sequential_only": fixed[1]=0; fixed[2]=0
        def rf(x): th=fixed.copy(); th[free]=x; return residual(th,reps,cond)
        out=least_squares(rf,fixed[free],max_nfev=800,xtol=1e-9,ftol=1e-9,gtol=1e-9); th=fixed.copy(); th[free]=out.x; res=residual(th,reps,cond); sse=float(np.sum(res**2)); k=len(free); rmse=np.sqrt(sse/n); sst=np.sum((obs-obs.mean())**2); r2=1-sse/sst; aic=n*np.log(max(sse/n,1e-300))+2*k; aicc=aic+2*k*(k+1)/(n-k-1); bic=n*np.log(max(sse/n,1e-300))+k*np.log(n); rows.append({"Condition_ID":cid,"Model":name,"RMSE_uM":rmse,"R2":r2,"AICc":aicc,"BIC":bic})
comparison=pd.DataFrame(rows); comparison["delta_AICc"]=comparison.groupby("Condition_ID")["AICc"].transform(lambda x:x-x.min()); comparison["delta_BIC"]=comparison.groupby("Condition_ID")["BIC"].transform(lambda x:x-x.min())
best=comparison.loc[comparison.groupby("Condition_ID")["AICc"].idxmin()]
model_summary=comparison.groupby("Model").agg(median_RMSE_uM=("RMSE_uM","median"),median_R2=("R2","median"),median_delta_AICc=("delta_AICc","median"),median_delta_BIC=("delta_BIC","median")).reset_index(); counts=best["Model"].value_counts().rename("AICc_best_count"); model_summary=model_summary.merge(counts,left_on="Model",right_index=True,how="left").fillna({"AICc_best_count":0}); display(model_summary.round(3))
fig,ax=plt.subplots(figsize=(8,4.4)); order=list(models); ax.bar(order,[int(model_summary.set_index("Model").loc[m,"AICc_best_count"]) for m in order]); ax.set_ylabel("Conditions with lowest AICc"); ax.set_title("Alternative kinetic schemes across 55 conditions"); ax.tick_params(axis='x',rotation=20); plt.show()

# Save all reviewer-analysis tables for optional SI use.
all_condition_CI.to_csv("reviewer_all_condition_parameter_CI.csv",index=False)
bootstrap.to_csv("reviewer_bootstrap_500_key_conditions.csv",index=False)
multistart.to_csv("reviewer_multistart_key_conditions.csv",index=False)
profile.to_csv("reviewer_profile_likelihood.csv",index=False)
comparison.to_csv("reviewer_alternative_model_comparison.csv",index=False)


## 8. Reproducibility outputs and manuscript checks

Export tabular results in machine-readable format. The supplementary notebook may contain more diagnostics than the manuscript/SI for transparency. **Do not replace the manuscript’s original student fitted constants with effective-rate fits**. The code should be fully executed on a local environment before claiming numerical reproduction of all supporting tables.

In [ ]:
from pathlib import Path
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)
for name, variable in {
    "python_effective_fits_55_conditions": fitted_results_table,
    "matlab_parameter_ci_55_conditions": all_condition_CI,
    "ci_includes_zero_summary": zero_table,
    "bootstrap_parameter_samples": bootstrap,
    "bootstrap_parameter_summary": bootstrap_summary,
    "branching_ratio_summary": ratio_table,
    "multistart_parameter_samples": multistart,
    "multistart_summary": multistart_summary,
    "alternative_pathway_comparison": comparison,
}.items():
    variable.to_csv(output_dir / f"{name}.csv", index=False)
print(f"Saved output tables to {output_dir.resolve()}")
